# Create Barncancerfonden Awards (Swedish Childhood Cancer Fund)

Creates Barncancerfonden awards from the fund's own published decision lists
("Beviljade ansökningar <CODE><YEAR>") on
https://www.barncancerfonden.se/for-forskare/forskningsanslag/beviljade-forskningsanslag-och-tjanster/
(current year plus the "Beviljade anslag genom åren" accordion back to 2014,
~228 xlsx/xls/PDF/docx files, six of them still hosted on the fund's domain but
no longer linked: PR2017, KP2017, KF2017, PL2017, ST2017, PR2019).
**2,199 awards, 2014-2026, SEK 1.71bn** (local run 2026-10-01).

**Source / coverage:**
- 1,959 rows carry the fund's application number (`Ans no`, e.g. `PR2025-0011`,
  `KP2016-0006`, `Rc2022-0001`) with Swedish title, administering organisation
  and department, applicant, years and granted amount (SEK).
- 240 rows come from the prose position lists (`Beviljade tjänster` TJ2014-2020,
  TJ2022/ST2022, NBCNS 2014/2016): name, institution, title, position type, no
  application number and no per-grant amount. They ship with synthetic keys
  `BCF-<TJ|NC><year>-<family>-<given>` (`-2` suffix for a second grant to the
  same person in one list).
- Not parsed (no text layer, outlined glyphs): forskningstjänster TJ2021 and
  2024, reseanslag Rb2019. IM2018 is an empty table on the site.
- SweCRIS does not carry Barncancerfonden (full scan 2026-10-01).

**Scope:** every Barncancerfonden call is kept (research-funder pathways rule,
Kyle 2026-10-01): project grants (PR, KP, MT/MTI, PL, EU co-funding, IM, OB,
NBCNS), programmes (TSK), start-up grants (FSU/UB), research positions (TJ, ST,
DK, PD, FT, HFT, KF) **and** the small researcher-support calls: travel grants
(Ra/Rb/Rc, `funding_type = travel`), symposia and guest lecturers (SY/GF,
`conference`), methodology studies and research courses (ME/FK, `training`).
Flagged for review rather than dropped.

**`funder_award_id` (§2.1.1):** the application number exactly as printed
(`PR2017-0082` form). Citing works write the same string; 339 of the 1,092
distinct existing citation stubs for F4320323766 (crossref_work.grants,
crossref_work_funders, europepmc_work_funders, datacite_work_funders) collapse
exactly onto these ids (383 after normalising dash/space variants, which do
not collapse). Position-list rows use the synthetic `BCF-...` keys (0 stubs).

**Dates:** the lists give only the call year (in the application number) and a
duration; `start_year` = call year, `start_date`/`end_date` NULL (JAES
precedent). **Amount:** granted amount in SEK as printed; the MTI2019
MedTech4Health co-funded lists ship the Barncancerfonden part only. Positions
(ST/DK/PD/FT/HFT/KF and the prose lists) publish no amount (NULL).

**Prerequisites:** run `scripts/local/barncancerfonden_to_s3.py` first. It
uploads `s3://openalex-ingest/awards/barncancerfonden/barncancerfonden_projects.parquet`
(§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320323766`,
Barncancerfonden (ror 05072yv34, doi 10.13039/501100006313), read from
`openalex.funders.funders` by id. No twin row.

**Priority:** `580` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.barncancerfonden_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/barncancerfonden/barncancerfonden_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows, COUNT(DISTINCT application_number) as distinct_ids
FROM openalex.awards.barncancerfonden_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.barncancerfonden_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.barncancerfonden_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320323766 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320323766;

## Step 2: Create Barncancerfonden Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.barncancerfonden_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323766  -- Barncancerfonden
),
src AS (
    SELECT
        TRIM(application_number) AS funder_award_id,
        NULLIF(TRIM(title), '') AS title,
        TRY_CAST(amount AS DOUBLE) AS amount,
        NULLIF(TRIM(currency), '') AS currency,
        NULLIF(TRIM(funding_type), '') AS funding_type,
        NULLIF(TRIM(funder_scheme), '') AS funder_scheme,
        TRY_CAST(call_year AS INT) AS call_year,
        NULLIF(TRIM(lead_given_name), '') AS given_name,
        NULLIF(TRIM(lead_family_name), '') AS family_name,
        NULLIF(TRIM(organisation), '') AS organisation,
        source_file
    FROM openalex.awards.barncancerfonden_raw
    WHERE application_number IS NOT NULL AND TRIM(application_number) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    g.title as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    CASE WHEN g.amount IS NOT NULL THEN COALESCE(g.currency, 'SEK') END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    COALESCE(g.funding_type, 'research') as funding_type,
    g.funder_scheme,
    'barncancerfonden_beviljade_anslag' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    g.call_year as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.family_name IS NOT NULL THEN named_struct(
        'given_name', g.given_name,
        'family_name', g.family_name,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.organisation,
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_file as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM src g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- §2.1.1 shape check: application numbers (PR2025-0011 style) or the
-- synthetic position keys; expect 0 rows of any other shape.
SELECT funder_award_id, display_name
FROM openalex.awards.barncancerfonden_awards
WHERE NOT (funder_award_id RLIKE '^[A-Za-z]{1,5}[0-9]{4}-[0-9]{3,4}$'
        OR funder_award_id RLIKE '^BCF-(TJ|NC)[0-9]{4}-[a-z0-9-]+$')
LIMIT 20;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'barncancerfonden_beviljade_anslag' AND priority = 580;

-- Priority 580: direct-from-funder ingest of Barncancerfonden's decision lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 580 outranks the
-- crossref/europepmc/datacite acknowledgement shells (priority 0) sharing these ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    580 as priority
FROM openalex.awards.barncancerfonden_awards
-- Kyle 2026-10-01: travel and conference/symposium grants are not research funding (Conquer Cancer/ASH rule): not shipped.
WHERE funding_type NOT IN ('travel', 'conference');

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.barncancerfonden_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.barncancerfonden_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_sek
FROM openalex.awards.barncancerfonden_awards
GROUP BY funding_type, funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_sek
FROM openalex.awards.barncancerfonden_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.barncancerfonden_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.barncancerfonden_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect ~100% title/PI, ~85% amount: positions publish none).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.barncancerfonden_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement stubs (expect ~339 shared ids).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as matched_by_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.barncancerfonden_awards t ON t.id = c.id
WHERE c.funder_id = 4320323766 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'barncancerfonden_beviljade_anslag'
GROUP BY provenance, priority;